Diurnal Asymmetry and Spatial Heterogeneity in Lake Daily Extreme Temperature Warming

Moving Windows Trend

In [ ]:
import rasterio
import numpy as np
import os
import re
from tqdm import tqdm

# ============================================================
# 1. Paths
# ============================================================

input_folder = r"F:\ERA5_Land\LMLT_season"
output_folder = r"F:\ERA5_Land\LMLT_season\Trend_Move_Summer"

os.makedirs(output_folder, exist_ok=True)


# ============================================================
# 2. Get Spring files
# ============================================================

tif_files = [
    f for f in os.listdir(input_folder)
    if f.lower().startswith("summer_") and f.lower().endswith(".tif")
]


# ============================================================
# 3. Sort by year
# ============================================================
tif_files.sort(key=lambda x: int(re.search(r'(\d{4})', x).group(1)))


# ============================================================
# 4. Read data
# ============================================================

data_list = []
profile = None

for file in tqdm(tif_files, desc="Reading files"):
    file_path = os.path.join(input_folder, file)
    with rasterio.open(file_path) as src:
        data_list.append(src.read(1))
        if profile is None:
            profile = src.profile.copy()


# ============================================================
# 5. Stack data
# ============================================================

data_stack = np.stack(data_list, axis=0)


# ============================================================
# 6. Calculate trend (Theil-Sen slope)
# ============================================================

n_years = data_stack.shape[0]
years = np.arange(n_years)

rows, cols = data_stack.shape[1], data_stack.shape[2]
slope = np.full((rows, cols), np.nan, dtype=np.float32)

for i in tqdm(range(rows), desc="Calculating trend"):
    for j in range(cols):
        ts = data_stack[:, i, j]
        mask = ~np.isnan(ts)
        if mask.sum() < 2:
            continue
        y = years[mask]
        v = ts[mask]
        slopes = []
        for m in range(len(v)):
            for n in range(m + 1, len(v)):
                slopes.append((v[n] - v[m]) / (y[n] - y[m]))
        slope[i, j] = np.median(slopes)


# ============================================================
# 7. Save result
# ============================================================

profile.update(dtype=rasterio.float32, count=1, nodata=np.nan)

output_path = os.path.join(output_folder, "trend_summer.tif")

with rasterio.open(output_path, "w", **profile) as dst:
    dst.write(slope, 1)

print("Done!")

Lake heat regions sstatistics

In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
from rasterio.mask import mask
import pandas as pd
import seaborn as sns

# ==================================================
# Global style settings
# ==================================================
sns.set_theme(style="white", font="Times New Roman")

matplotlib.rcParams['axes.unicode_minus'] = False

matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif'] = ['Times New Roman']

# ==================================================
# File paths
# ==================================================
season_files = {
    "Spring": r"F:\Reversed_asymmetric_warming_inlake\DTR_Average\Spring_DTR.tif",
    "Summer": r"F:\Reversed_asymmetric_warming_inlake\DTR_Average\Summer_DTR.tif",
    "Autumn": r"F:\Reversed_asymmetric_warming_inlake\DTR_Average\Autumn_DTR.tif",
    "Winter": r"F:\Reversed_asymmetric_warming_inlake\DTR_Average\Winter_DTR.tif"
}

shp_path = r"E:\Data\lake_thermal_regions1.shp"

# ==================================================
# Real global season mapping
# ==================================================
season_map = {
    "Spring": {
        "North": "Spring",
        "South": "Autumn"
    },
    "Summer": {
        "North": "Summer",
        "South": "Winter"
    },
    "Autumn": {
        "North": "Autumn",
        "South": "Spring"
    },
    "Winter": {
        "North": "Winter",
        "South": "Summer"
    }
}

# ==================================================
# Read thermal region shapefile
# ==================================================
gdf_lakes = gpd.read_file(shp_path)

print("Lake thermal regions:")
print(gdf_lakes.head())

print("\nRegions:")
print(gdf_lakes["regions"].unique())

# ==================================================
# Open all seasonal rasters
# ==================================================
srcs = {}

for season, path in season_files.items():

    print(f"Opening {season}")

    srcs[season] = rasterio.open(path)

# ==================================================
# Extract north/south hemisphere pixels within region
# ==================================================
def extract_hemisphere_pixels(region_gdf, src):

    shapes = region_gdf.geometry.values

    out_image, out_transform = mask(
        src,
        shapes,
        crop=True
    )

    data = out_image[0]

    nodata = src.nodata

    if nodata is None:
        nodata = -9999

    valid_mask = (
        (data != nodata)
        & np.isfinite(data)
        & (data > -1000)
    )

    rows, cols = np.where(valid_mask)

    if len(rows) == 0:

        return (
            np.array([]),
            np.array([])
        )

    xs, ys = rasterio.transform.xy(
        out_transform,
        rows,
        cols
    )

    latitudes = np.array(ys)

    values = data[rows, cols]

    north_values = values[
        latitudes >= 0
    ]

    south_values = values[
        latitudes < 0
    ]

    return (
        north_values,
        south_values
    )

# ==================================================
# Statistics for real global seasons
# ==================================================
def process_global_season(
        season_name,
        gdf_lakes,
        srcs):

    region_values = {}

    north_src = srcs[
        season_map[season_name]["North"]
    ]

    south_src = srcs[
        season_map[season_name]["South"]
    ]

    for region in gdf_lakes["regions"].unique():

        print(
            f"{season_name} | {region}"
        )

        region_gdf = gdf_lakes[
            gdf_lakes["regions"] == region
        ]

        # Northern hemisphere part
        north_values, _ = extract_hemisphere_pixels(
            region_gdf,
            north_src
        )

        # Southern hemisphere part
        _, south_values = extract_hemisphere_pixels(
            region_gdf,
            south_src
        )

        merged = np.concatenate(
            [
                north_values,
                south_values
            ]
        )

        if len(merged) > 0:

            region_values[region] = merged

        else:

            region_values[region] = np.array([])

        print(
            f"Pixels = {len(merged)}"
        )

    return region_values

# ==================================================
# Boxplot
# ==================================================
def plot_boxplot(
        region_values,
        season_name):

    valid_region_values = {
        k: v
        for k, v in region_values.items()
        if len(v) > 0
    }

    if len(valid_region_values) == 0:

        print(
            f"No valid data for {season_name}"
        )

        return

    max_len = max(
        len(v)
        for v in valid_region_values.values()
    )

    padded_values = {}

    for region, values in valid_region_values.items():

        padded_values[region] = np.pad(
            values,
            (
                0,
                max_len - len(values)
            ),
            constant_values=np.nan
        )

    data = pd.DataFrame(
        padded_values
    )

    fig, ax = plt.subplots(
        figsize=(12, 7)
    )

    sns.boxplot(
        data=data * 10,
        width=0.5,
        linewidth=1.8,
        fliersize=5,
        palette="Set2",
        ax=ax
    )

    means = data.mean(axis=0) * 10

    ax.scatter(
        range(len(data.columns)),
        means,
        color='white',
        edgecolor='black',
        s=120,
        zorder=3,
        label='Mean'
    )

    ax.set_xticklabels(
        ax.get_xticklabels(),
        fontsize=26
    )

    ax.set_yticklabels(
        ax.get_yticklabels(),
        fontsize=26
    )

    ax.set_ylabel(
        r'Trend in DTR (°C decade$^{-1}$)',
        fontsize=30
    )

    ax.set_title(
        season_name,
        fontsize=30
    )

    sns.despine()

    ax.spines['left'].set_linewidth(1.2)
    ax.spines['bottom'].set_linewidth(1.2)

    ax.tick_params(
        axis='both',
        which='major',
        length=6,
        width=1.2
    )

    ax.legend(
        frameon=False,
        fontsize=14
    )

    plt.tight_layout()

    plt.show()

# ==================================================
# Process four real global seasons sequentially
# ==================================================
for season in [
    "Spring",
    "Summer",
    "Autumn",
    "Winter"
]:

    print(
        f"\nProcessing {season}"
    )

    region_values = process_global_season(
        season,
        gdf_lakes,
        srcs
    )

    plot_boxplot(
        region_values,
        season
    )

# ==================================================
# Close rasters
# ==================================================
for src in srcs.values():
    src.close()

Sensonality analysis

In [ ]:
# ==========================================================
# Global lake DTR seasonal trend statistics
#
# Season data named by Northern Hemisphere:
# Spring_DTR.tif
# Summer_DTR.tif
# Autumn_DTR.tif
# Winter_DTR.tif
#
# Remapped by hemisphere:
#
# Global Spring = NH Spring + SH Autumn
# Global Summer = NH Summer + SH Winter
# Global Autumn = NH Autumn + SH Spring
# Global Winter = NH Winter + SH Summer
#
# Classes:
# Glacier / City / Other
#
# * Chunked reading to reduce VS Code Kernel crash risk
# ==========================================================

import rasterio
import numpy as np
import geopandas as gpd
from rasterio.features import rasterize
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
import gc
import os

# ==========================================================
# 0. Matplotlib settings
# ==========================================================

matplotlib.rcParams['font.family'] = 'Times New Roman'
matplotlib.rcParams['font.size'] = 36

plt.close('all')

# ==========================================================
# 1. File paths
# ==========================================================

base_dir = r"F:\Reversed_asymmetric_warming_inlake\min_T_monthly"

urban_shp_path = (
    r"F:\Reversed_asymmetric_warming_inlake"
    r"\GUB\GUB_Global_2018.shp"
)

glacier_path = (
    r"E:\Data\glacier_area\glacier\BC.shp"
)

# ==========================================================
# 2. Four seasonal files
# ==========================================================

season_files = {
    'Spring': os.path.join(
        base_dir, 'Spring_min_average.tif'
    ),

    'Summer': os.path.join(
        base_dir, 'Summer_min_average.tif'
    ),

    'Autumn': os.path.join(
        base_dir, 'Autumn_min_average.tif'
    ),

    'Winter': os.path.join(
        base_dir, 'Winter_min_average.tif'
    )
}

# ==========================================================
# 3. Hemisphere season mapping
#
# Final seasons:
#
# Spring = NH Spring + SH Autumn
# Summer = NH Summer + SH Winter
# Autumn = NH Autumn + SH Spring
# Winter = NH Winter + SH Summer
# ==========================================================

season_mapping = {

    'Spring': {
        'NH': 'Spring',
        'SH': 'Autumn'
    },

    'Summer': {
        'NH': 'Summer',
        'SH': 'Winter'
    },

    'Autumn': {
        'NH': 'Autumn',
        'SH': 'Spring'
    },

    'Winter': {
        'NH': 'Winter',
        'SH': 'Summer'
    }
}

# ==========================================================
# 4. Chunk size
# ==========================================================

CHUNK_SIZE = 512

# ==========================================================
# 5. Read basic info from first raster
# ==========================================================

reference_file = season_files['Spring']

with rasterio.open(reference_file) as src:

    height = src.height
    width = src.width
    transform = src.transform
    crs = src.crs

print("=" * 70)
print("Global lake DTR seasonal trend statistics")
print("=" * 70)

print(f"Raster size: {width} x {height}")
print(f"CRS: {crs}")

# ==========================================================
# 6. Create global latitude mask
#
# * Use pixel center latitude
#
# latitude >= 0 -> Northern Hemisphere
# latitude <  0 -> Southern Hemisphere
# ==========================================================

rows = np.arange(height)

latitudes = (
    transform.f +
    (rows + 0.5) * transform.e
)

north_mask_1d = latitudes >= 0
south_mask_1d = latitudes < 0

print()
print("Hemisphere split complete")

print(
    f"Northern Hemisphere rows: {np.sum(north_mask_1d):,}"
)

print(
    f"Southern Hemisphere rows: {np.sum(south_mask_1d):,}"
)

# ==========================================================
# 7. Read urban data
# ==========================================================

print()
print("Reading urban data...")

gdf_urban = gpd.read_file(
    urban_shp_path
)

print(
    f"Urban feature count: {len(gdf_urban):,}"
)

if gdf_urban.crs != crs:

    gdf_urban = gdf_urban.to_crs(crs)

# ==========================================================
# 8. Rasterize urban areas
# ==========================================================

print("Rasterizing urban areas...")

urban_shapes = (
    (geom, 1)
    for geom in gdf_urban.geometry
    if geom is not None
    and not geom.is_empty
)

urban_mask = rasterize(
    urban_shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    all_touched=True,
    dtype=np.uint8
).astype(bool)

del gdf_urban
del urban_shapes

gc.collect()

print("Urban rasterization complete")

# ==========================================================
# 9. Read glacier data
# ==========================================================

print()
print("Reading glacier data...")

gdf_glacier = gpd.read_file(
    glacier_path
)

print(
    f"Glacier feature count: {len(gdf_glacier):,}"
)

if gdf_glacier.crs != crs:

    gdf_glacier = gdf_glacier.to_crs(crs)

# ==========================================================
# 10. Rasterize glacier areas
# ==========================================================

print("Rasterizing glacier areas...")

glacier_shapes = (
    (geom, 1)
    for geom in gdf_glacier.geometry
    if geom is not None
    and not geom.is_empty
)

glacier_mask = rasterize(
    glacier_shapes,
    out_shape=(height, width),
    transform=transform,
    fill=0,
    all_touched=True,
    dtype=np.uint8
).astype(bool)

del gdf_glacier
del glacier_shapes

gc.collect()

print("Glacier rasterization complete")

# ==========================================================
# 11. Online statistics accumulator
#
# Does not store all pixels
# Only stores:
# N
# Mean
# M2
# ==========================================================

class OnlineStats:

    def __init__(self):

        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0

    def update(self, values):

        if values.size == 0:
            return

        values = values.astype(
            np.float64,
            copy=False
        )

        n2 = values.size

        mean2 = np.mean(values)

        M2_2 = np.sum(
            (values - mean2) ** 2,
            dtype=np.float64
        )

        if self.n == 0:

            self.n = n2
            self.mean = mean2
            self.M2 = M2_2

            return

        n1 = self.n
        mean1 = self.mean
        M2_1 = self.M2

        delta = mean2 - mean1

        n_total = n1 + n2

        self.mean = (
            mean1 +
            delta * n2 / n_total
        )

        self.M2 = (
            M2_1 +
            M2_2 +
            delta ** 2 *
            n1 * n2 / n_total
        )

        self.n = n_total

    def std(self):

        if self.n == 0:
            return np.nan

        return np.sqrt(
            self.M2 / self.n
        )


# ==========================================================
# 12. Statistics function
# ==========================================================

def process_season(
    target_season,
    nh_season,
    sh_season
):

    print()
    print("=" * 70)

    print(
        f"Computing global {target_season}"
    )

    print(
        f"Northern Hemisphere: {nh_season}_DTR.tif"
    )

    print(
        f"Southern Hemisphere: {sh_season}_DTR.tif"
    )

    print("=" * 70)

    # ------------------------------------------------------
    # Three class accumulators
    # ------------------------------------------------------

    glacier_stats = OnlineStats()
    city_stats = OnlineStats()
    other_stats = OnlineStats()

    # ------------------------------------------------------
    # Process Northern and Southern Hemispheres separately
    # ------------------------------------------------------

    hemisphere_info = [

        (
            'NH',
            season_files[nh_season],
            north_mask_1d
        ),

        (
            'SH',
            season_files[sh_season],
            south_mask_1d
        )
    ]

    # ======================================================
    # Process NH and SH sequentially
    # ======================================================

    for hemisphere, tif_path, hemisphere_mask in hemisphere_info:

        print()
        print(
            f"Processing {hemisphere}: "
            f"{os.path.basename(tif_path)}"
        )

        # --------------------------------------------------
        # Open current seasonal raster
        # --------------------------------------------------

        with rasterio.open(tif_path) as src:

            # Check raster size
            if (
                src.height != height
                or src.width != width
            ):

                raise ValueError(
                    f"{tif_path} size does not match reference raster!"
                )

            # ------------------------------------------------
            # Chunked processing
            # ------------------------------------------------

            total_blocks = (
                (height + CHUNK_SIZE - 1)
                // CHUNK_SIZE
            ) * (
                (width + CHUNK_SIZE - 1)
                // CHUNK_SIZE
            )

            block_counter = 0

            for row_start in range(
                0,
                height,
                CHUNK_SIZE
            ):

                row_end = min(
                    row_start + CHUNK_SIZE,
                    height
                )

                # --------------------------------------------
                # Whether current rows belong to target hemisphere
                # --------------------------------------------

                hemisphere_rows = (
                    hemisphere_mask[
                        row_start:row_end
                    ]
                )

                # --------------------------------------------
                # If this whole row block does not belong to target hemisphere
                # skip directly
                # --------------------------------------------

                if not np.any(
                    hemisphere_rows
                ):

                    continue

                for col_start in range(
                    0,
                    width,
                    CHUNK_SIZE
                ):

                    col_end = min(
                        col_start + CHUNK_SIZE,
                        width
                    )

                    block_counter += 1

                    # ----------------------------------------
                    # Current window
                    # ----------------------------------------

                    window = rasterio.windows.Window(
                        col_start,
                        row_start,
                        col_end - col_start,
                        row_end - row_start
                    )

                    # ----------------------------------------
                    # Read data
                    # ----------------------------------------

                    block = src.read(
                        1,
                        window=window
                    ).astype(
                        np.float32,
                        copy=False
                    )

                    # ----------------------------------------
                    # Current hemisphere mask
                    # ----------------------------------------

                    hemi_mask = np.broadcast_to(
                        hemisphere_rows[:, None],
                        block.shape
                    )

                    # ----------------------------------------
                    # Urban / glacier mask
                    # ----------------------------------------

                    urban_block = (
                        urban_mask[
                            row_start:row_end,
                            col_start:col_end
                        ]
                    )

                    glacier_block = (
                        glacier_mask[
                            row_start:row_end,
                            col_start:col_end
                        ]
                    )

                    # ----------------------------------------
                    # Valid values
                    # ----------------------------------------

                    valid = (
                        np.isfinite(block)
                        &
                        (block != 0)
                        &
                        (block >= -1000)
                        &
                        hemi_mask
                    )

                    # ----------------------------------------
                    # Consistent with original code: x10
                    # ----------------------------------------

                    block *= 10

                    # ========================================
                    # Glacier
                    # ========================================

                    mask_g = (
                        valid &
                        glacier_block
                    )

                    if np.any(mask_g):

                        values_g = block[mask_g]

                        glacier_stats.update(
                            values_g
                        )

                        del values_g

                    # ========================================
                    # City
                    # ========================================

                    mask_c = (
                        valid &
                        urban_block
                    )

                    if np.any(mask_c):

                        values_c = block[mask_c]

                        city_stats.update(
                            values_c
                        )

                        del values_c

                    # ========================================
                    # Other
                    # ========================================

                    mask_o = (
                        valid &
                        (~glacier_block) &
                        (~urban_block)
                    )

                    if np.any(mask_o):

                        values_o = block[mask_o]

                        other_stats.update(
                            values_o
                        )

                        del values_o

                    # ----------------------------------------
                    # Clean up temporary variables
                    # ----------------------------------------

                    del block
                    del valid
                    del hemi_mask
                    del mask_g
                    del mask_c
                    del mask_o

                    # ----------------------------------------
                    # Periodic garbage collection
                    # ----------------------------------------

                    if block_counter % 50 == 0:

                        gc.collect()

            print(
                f"{hemisphere} processing complete"
            )

    # ======================================================
    # Get statistics results
    # ======================================================

    result = {

        'Glacier': {
            'N': glacier_stats.n,
            'mean': glacier_stats.mean,
            'std': glacier_stats.std()
        },

        'City': {
            'N': city_stats.n,
            'mean': city_stats.mean,
            'std': city_stats.std()
        },

        'Other': {
            'N': other_stats.n,
            'mean': other_stats.mean,
            'std': other_stats.std()
        }
    }

    # ======================================================
    # ANOVA
    #
    # Computed directly from N / Mean / M2
    # Avoids storing all pixels
    # ======================================================

    groups = [
        glacier_stats,
        city_stats,
        other_stats
    ]

    N_total = sum(
        g.n for g in groups
    )

    k = len(groups)

    if N_total > k:

        # Grand mean
        grand_mean = (
            sum(
                g.n * g.mean
                for g in groups
            )
            / N_total
        )

        # Between-group sum of squares
        SS_between = sum(
            g.n *
            (g.mean - grand_mean) ** 2
            for g in groups
        )

        # Within-group sum of squares
        SS_within = sum(
            g.M2
            for g in groups
        )

        df_between = k - 1
        df_within = N_total - k

        MS_between = (
            SS_between /
            df_between
        )

        MS_within = (
            SS_within /
            df_within
        )

        F_stat = (
            MS_between /
            MS_within
        )

        p_val = stats.f.sf(
            F_stat,
            df_between,
            df_within
        )

    else:

        F_stat = np.nan
        p_val = np.nan

    # ======================================================
    # Output
    # ======================================================

    print()
    print(
        f"[Global {target_season}]"
    )

    print(
        f"Glacier: "
        f"N = {glacier_stats.n:,}, "
        f"Mean = {glacier_stats.mean:.4f}, "
        f"SD = {glacier_stats.std():.4f}"
    )

    print(
        f"City:    "
        f"N = {city_stats.n:,}, "
        f"Mean = {city_stats.mean:.4f}, "
        f"SD = {city_stats.std():.4f}"
    )

    print(
        f"Other:   "
        f"N = {other_stats.n:,}, "
        f"Mean = {other_stats.mean:.4f}, "
        f"SD = {other_stats.std():.4f}"
    )

    print(
        f"ANOVA: "
        f"F = {F_stat:.6f}, "
        f"p = {p_val:.6e}"
    )

    # ======================================================
    # Return
    # ======================================================

    return {
        'season': target_season,

        'Glacier_mean':
            glacier_stats.mean,

        'Glacier_std':
            glacier_stats.std(),

        'Glacier_N':
            glacier_stats.n,

        'City_mean':
            city_stats.mean,

        'City_std':
            city_stats.std(),

        'City_N':
            city_stats.n,

        'Other_mean':
            other_stats.mean,

        'Other_std':
            other_stats.std(),

        'Other_N':
            other_stats.n,

        'F':
            F_stat,

        'p':
            p_val
    }


# ==========================================================
# 13. Start statistics for four global real seasons
# ==========================================================

all_results = []

for target_season in [
    'Spring',
    'Summer',
    'Autumn',
    'Winter'
]:

    mapping = season_mapping[
        target_season
    ]

    result = process_season(
        target_season,
        mapping['NH'],
        mapping['SH']
    )

    all_results.append(
        result
    )

    gc.collect()


# ==========================================================
# 14. Summary results
# ==========================================================

print()
print()
print("=" * 90)
print("Global four-season DTR trend statistics summary")
print("=" * 90)

for r in all_results:

    print()
    print(
        f"===== {r['season']} ====="
    )

    print(
        f"Glacier: "
        f"{r['Glacier_mean']:.4f} "
        f"+/- {r['Glacier_std']:.4f}"
    )

    print(
        f"City:    "
        f"{r['City_mean']:.4f} "
        f"+/- {r['City_std']:.4f}"
    )

    print(
        f"Other:   "
        f"{r['Other_mean']:.4f} "
        f"+/- {r['Other_std']:.4f}"
    )

    print(
        f"ANOVA p = "
        f"{r['p']:.6e}"
    )


# ==========================================================
# 15. Plot four seasons
# ==========================================================

labels = [
    'Glacier',
    'City',
    'Other'
]

for r in all_results:

    season = r['season']

    means = [
        r['Glacier_mean'],
        r['City_mean'],
        r['Other_mean']
    ]

    stds = [
        r['Glacier_std'],
        r['City_std'],
        r['Other_std']
    ]

    # ------------------------------------------------------
    # Create figure
    # ------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(8, 6)
    )

    bars = ax.bar(
        labels,
        means,
        yerr=stds,
        capsize=5,
        color=[
            '#4DBBD5',
            '#FFB74D',
            '#81C784'
        ],
        edgecolor='black'
    )

    # ------------------------------------------------------
    # Values on bars
    # ------------------------------------------------------

    for i, bar in enumerate(bars):

        height_bar = bar.get_height()

        if np.isfinite(height_bar):

            ax.text(
                bar.get_x()
                + bar.get_width() / 2,

                height_bar + 0.001,

                f'{means[i]:.2f}',

                ha='center',
                va='bottom',

                fontsize=36
            )

    # ------------------------------------------------------
    # Zero line
    # ------------------------------------------------------

    ax.axhline(
        0,
        color='black',
        linewidth=1
    )

    # ------------------------------------------------------
    # Y axis
    # ------------------------------------------------------

    ax.set_ylabel(
        'Trend in Min (°C decade⁻¹)',
        fontsize=30
    )

    # ------------------------------------------------------
    # Ticks
    # ------------------------------------------------------

    ax.tick_params(
        axis='both',
        labelsize=34
    )

    # ------------------------------------------------------
    # Grid
    # ------------------------------------------------------

    ax.yaxis.grid(
        True,
        linestyle='--',
        alpha=0.3
    )

    # ------------------------------------------------------
    # Layout
    # ------------------------------------------------------

    plt.tight_layout()

    # ------------------------------------------------------
    # Save
    # ------------------------------------------------------

    output_path = os.path.join(
        base_dir,
        f'F:\Reversed_asymmetric_warming_inlake\{season}_min_Global_Glacier_City_Other.png'
    )

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )

    plt.close(fig)

    print()
    print(
        f"{season} figure saved:"
    )

    print(output_path)

    gc.collect()


# ==========================================================
# 16. Final completion
# ==========================================================

print()
print("=" * 90)
print("All four global real seasons statistics complete!")
print("=" * 90)

print()
print("Season mapping:")

print(
    "Global Spring = NH Spring + SH Autumn"
)

print(
    "Global Summer = NH Summer + SH Winter"
)

print(
    "Global Autumn = NH Autumn + SH Spring"
)

print(
    "Global Winter = NH Winter + SH Summer"
)

print()
print("Program finished.")

Random forest

In [ ]:
# ============================================================
# Seasonal Random Forest Attribution
# 1950-2022
#
# Research objective:
#   Use 30-year moving trends from 1950-2022
#   to explain spatial differences in seasonal LSWT maximum
#   warming trends in the Northern Hemisphere
#
# Data processing approach:
#   Follow the successfully running Year RF code
#
# Core:
#   1. Use each season's Y as reference raster
#   2. Reproject clip_1.tif to Y grid
#   3. Reproject all driver variables to Y grid
#   4. Keep only Northern Hemisphere
#   5. Remove NaN uniformly
#   6. StandardScaler
#   7. Random Forest
#   8. Save Feature Importance
#   9. Save model, Scaler, X/Y and test set
#
# Note:
#   LakeIceDuration:
#       F:\Lake_Ice_Duration_Trend_NH.tif
#       All four seasons use exactly the same data
#
#   LakeDepth:
#       F:\ERA5_Land\Lake_depth.tif
#       All four seasons use exactly the same data
#
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import os
import gc
import warnings
import traceback

import numpy as np
import pandas as pd
import rasterio

from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import xy

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from scipy.stats import pearsonr

import joblib


warnings.filterwarnings("ignore")


# ============================================================
# 1. Basic parameters
# ============================================================

START_YEAR = 1950
END_YEAR = 2022

SEASONS = [
    "Spring",
    "Summer",
    "Autumn",
    "Winter"
]


# ============================================================
# 2. Random Forest parameters
# ============================================================

N_ESTIMATORS = 300

MAX_DEPTH = 20

MIN_SAMPLES_LEAF = 2

TEST_SIZE = 0.30

RANDOM_STATE = 42

# To avoid Jupyter Kernel crashing due to excessive parallelism,
# four seasons are not run in parallel here.
RF_THREADS = -1


# ============================================================
# 3. Output paths
# ============================================================

base_out = (
    r"F:\Reversed_asymmetric_warming_inlake"
    r"\outputs_seasonal_RF_max_test"
)

os.makedirs(
    base_out,
    exist_ok=True
)


print("=" * 90)
print("Seasonal Random Forest Attribution")
print("=" * 90)

print(
    f"Years            : {START_YEAR}-{END_YEAR}"
)

print(
    "Seasons          : Spring / Summer / Autumn / Winter"
)

print(
    "Spatial extent   : Northern Hemisphere"
)

print(
    f"RF trees         : {N_ESTIMATORS}"
)

print(
    f"RF max_depth     : {MAX_DEPTH}"
)

print(
    f"RF min_samples_leaf : {MIN_SAMPLES_LEAF}"
)

print(
    "Season mode      : Sequential"
)

print("=" * 90)


# ============================================================
# 4. Input paths
# ============================================================


# ------------------------------------------------------------
# 4.1 ERA5 seasonal trend file directory
# ------------------------------------------------------------
#
# Actual file structure:
#
# F:\ERA5_Land\Seasonal_MovingTrend_Mean\
#
# Evaporation_Spring_MovingTrendMean_30yr.tif
# Evaporation_Summer_MovingTrendMean_30yr.tif
# ...
#
# Precip_Spring_MovingTrendMean_30yr.tif
# ...
#
# ------------------------------------------------------------

trend_base_dir = (
    r"F:\ERA5_Land\Seasonal_MovingTrend_Mean"
)


if not os.path.exists(trend_base_dir):

    raise FileNotFoundError(
        f"\nSeasonal trend directory does not exist:\n"
        f"{trend_base_dir}"
    )


# ------------------------------------------------------------
# 4.2 Y data
# ------------------------------------------------------------

y_files = {

    "Spring":
        r"F:\Reversed_asymmetric_warming_inlake"
        r"\max_T_monthly\Spring_max_average.tif",

    "Summer":
        r"F:\Reversed_asymmetric_warming_inlake"
        r"\max_T_monthly\Summer_max_average_clip.tif",

    "Autumn":
        r"F:\Reversed_asymmetric_warming_inlake"
        r"\max_T_monthly\Autumn_max_average.tif",

    "Winter":
        r"F:\Reversed_asymmetric_warming_inlake"
        r"\max_T_monthly\Winter_max_average.tif"
}


# ------------------------------------------------------------
# 4.3 Lake mask
# ------------------------------------------------------------

mask_path = (
    r"F:\Reversed_asymmetric_warming_inlake"
    r"\clip_1.tif"
)


# ------------------------------------------------------------
# 4.4 Lake depth
# ------------------------------------------------------------

lake_depth_file = (
    r"F:\ERA5_Land\Lake_depth.tif"
)


# ------------------------------------------------------------
# 4.5 Lake ice duration trend
# ------------------------------------------------------------
#
# All four seasons use exactly the same trend raster
#
# ------------------------------------------------------------

lake_ice_duration_file = (
    r"F:\Lake_Ice_Duration_Trend_NH.tif"
)


# ============================================================
# 5. Feature names
# ============================================================

feature_names = [

    "Evaporation",

    "LCC",

    "SP",

    "SSRD",

    "STRD",

    "Temp_2m",

    "Precip",

    "U10",

    "V10",

    "LMLD",

    "LakeIceDuration",

    "LakeDepth"

]


print("\nDriver variables:")

for i, feature in enumerate(
    feature_names,
    start=1
):

    print(
        f"{i:02d}. {feature}"
    )


# ============================================================
# 6. Check fixed variables
# ============================================================

print("\n")
print("=" * 90)
print("Checking fixed variables")
print("=" * 90)


if not os.path.exists(
    lake_depth_file
):

    raise FileNotFoundError(
        f"LakeDepth file does not exist:\n"
        f"{lake_depth_file}"
    )


if not os.path.exists(
    lake_ice_duration_file
):

    raise FileNotFoundError(
        f"LakeIceDuration file does not exist:\n"
        f"{lake_ice_duration_file}"
    )


if not os.path.exists(
    mask_path
):

    raise FileNotFoundError(
        f"Mask file does not exist:\n"
        f"{mask_path}"
    )


print(
    "LakeDepth       : OK"
)

print(
    "LakeIceDuration : OK"
)

print(
    "Mask            : OK"
)


# ============================================================
# 7. Check Y files
# ============================================================

print("\nChecking Y files:")

for season in SEASONS:

    path = y_files[season]

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\n{season} Y file does not exist:\n"
            f"{path}"
        )

    print(
        f"{season:10s}: OK"
    )


# ============================================================
# 8. Utility function:
#    determine whether Northern Hemisphere
# ============================================================

def create_northern_mask(
    transform,
    crs,
    rows,
    cols
):
    """
    Based on the reference raster's spatial reference,
    create a latitude >= 0 Northern Hemisphere mask.

    This study assumes the reference raster is in
    a geographic coordinate system.
    """

    if crs is None:

        raise ValueError(
            "Reference raster has no CRS; cannot determine latitude."
        )


    if not crs.is_geographic:

        raise ValueError(
            "\nReference raster is not in a geographic CRS.\n"
            f"CRS = {crs}\n\n"
            "Current code requires the reference Y raster to use "
            "a geographic coordinate system."
        )


    row_indices = np.arange(
        rows,
        dtype=np.float64
    )


    # Only compute latitudes for the middle column.
    # For a standard north-up geographic raster,
    # latitude is the same within a row.

    center_col = cols // 2


    _, latitudes = xy(

        transform,

        row_indices,

        np.full(
            rows,
            center_col
        )
    )


    latitudes = np.asarray(
        latitudes,
        dtype=np.float64
    )


    northern_mask = (
        latitudes >= 0
    )[:, None]


    northern_mask = np.broadcast_to(
        northern_mask,
        (rows, cols)
    )


    return northern_mask


# ============================================================
# 9. Raster reprojection / resampling function
# ============================================================

def read_and_match(
    path,
    rows,
    cols,
    ref_transform,
    ref_crs,
    resampling=Resampling.bilinear
):
    """
    Reproject/resample any raster to the reference Y grid.
    """

    if not os.path.exists(path):

        raise FileNotFoundError(
            f"\nInput file does not exist:\n{path}"
        )


    with rasterio.open(path) as src:

        dst = np.full(
            (rows, cols),
            np.nan,
            dtype=np.float32
        )


        # ----------------------------------------------------
        # Original nodata
        # ----------------------------------------------------

        src_nodata = src.nodata


        reproject(

            source=rasterio.band(
                src,
                1
            ),

            destination=dst,

            src_transform=src.transform,

            src_crs=src.crs,

            dst_transform=ref_transform,

            dst_crs=ref_crs,

            src_nodata=src_nodata,

            dst_nodata=np.nan,

            resampling=resampling

        )


    return dst


# ============================================================
# 10. Read one season
# ============================================================

def run_season(season):


    print("\n\n")
    print("#" * 90)
    print(
        f"Starting {season.upper()}"
    )
    print("#" * 90)


    # ========================================================
    # 10.1 Output directory
    # ========================================================

    season_out = os.path.join(
        base_out,
        season.lower()
    )


    os.makedirs(
        season_out,
        exist_ok=True
    )


    # ========================================================
    # 10.2 Read Y
    # ========================================================

    y_file = y_files[season]


    print(
        f"\n[{season}] Reading Y:"
    )

    print(
        y_file
    )


    with rasterio.open(
        y_file
    ) as src:

        y_data = src.read(
            1
        ).astype(
            np.float32
        )


        if src.nodata is not None:

            y_data[
                y_data == src.nodata
            ] = np.nan


        # Current season Y is used as its own reference grid

        ref_transform = src.transform

        ref_crs = src.crs

        rows = src.height

        cols = src.width


    print(
        f"Reference grid: "
        f"{rows} x {cols}"
    )


    print(
        f"CRS: {ref_crs}"
    )


    # ========================================================
    # 10.3 Create Northern Hemisphere mask
    # ========================================================

    print(
        f"\n[{season}] Creating Northern Hemisphere mask..."
    )


    north_mask = create_northern_mask(

        ref_transform,

        ref_crs,

        rows,

        cols
    )


    print(
        "Northern Hemisphere pixels:",
        int(
            north_mask.sum()
        )
    )


    # ========================================================
    # 10.4 Read lake mask
    # ========================================================

    print(
        f"\n[{season}] Reading lake mask..."
    )


    mask_match = read_and_match(

        mask_path,

        rows,

        cols,

        ref_transform,

        ref_crs,

        Resampling.nearest
    )


    lake_mask = (

        np.isfinite(mask_match)

        &

        (mask_match != 0)

        &

        north_mask

    )


    print(
        "Lake + Northern Hemisphere valid pixels:",
        int(
            lake_mask.sum()
        )
    )


    del mask_match
    del north_mask


    gc.collect()


    # ========================================================
    # 10.5 Apply mask to Y
    # ========================================================

    y_data[
        ~lake_mask
    ] = np.nan


    print(
        "Y valid pixels:",
        int(
            np.isfinite(y_data).sum()
        )
    )


    # ========================================================
    # 10.6 Read 12 X variables
    # ========================================================

    print(
        f"\n[{season}] Reading driver variables..."
    )


    X_arrays = []


    for feature in feature_names:


        print(
            "\n----------------------------------------"
        )

        print(
            f"{feature}"
        )


        # ====================================================
        # 10.6.1 LakeIceDuration
        # Same for all four seasons
        # ====================================================

        if feature == "LakeIceDuration":

            feature_file = (
                lake_ice_duration_file
            )


            print(
                "Type: fixed variable"
            )


            print(
                "Shared by all seasons:"
                f"\n{feature_file}"
            )


            arr = read_and_match(

                feature_file,

                rows,

                cols,

                ref_transform,

                ref_crs,

                Resampling.bilinear
            )


        # ====================================================
        # 10.6.2 LakeDepth
        # Same for all four seasons
        # ====================================================

        elif feature == "LakeDepth":

            feature_file = (
                lake_depth_file
            )


            print(
                "Type: fixed variable"
            )


            print(
                "Shared by all seasons:"
                f"\n{feature_file}"
            )


            arr = read_and_match(

                feature_file,

                rows,

                cols,

                ref_transform,

                ref_crs,

                Resampling.bilinear
            )


        # ====================================================
        # 10.6.3 ERA5 seasonal variables
        # ====================================================

        else:

            feature_file = os.path.join(

                trend_base_dir,

                f"{feature}_"
                f"{season}_"
                f"MovingTrendMean_30yr.tif"

            )


            print(
                "Type: seasonal trend variable"
            )


            print(
                "File:"
            )

            print(
                feature_file
            )


            if not os.path.exists(
                feature_file
            ):

                raise FileNotFoundError(

                    f"\nTrend file not found:\n"
                    f"{feature_file}\n\n"

                    f"Variable: {feature}\n"
                    f"Season: {season}\n"
                )


            arr = read_and_match(

                feature_file,

                rows,

                cols,

                ref_transform,

                ref_crs,

                Resampling.bilinear
            )


        # ====================================================
        # 10.6.4 Apply mask
        # ====================================================

        arr[
            ~lake_mask
        ] = np.nan


        finite_count = int(
            np.isfinite(arr).sum()
        )


        print(
            f"Valid pixels: {finite_count:,}"
        )


        # ====================================================
        # 10.6.5 Save
        # ====================================================

        X_arrays.append(
            arr
        )


        del arr

        gc.collect()


    # ========================================================
    # 10.7 Build X_stack
    # ========================================================

    print(
        f"\n[{season}] Building X_stack..."
    )


    X_stack = np.stack(
        X_arrays,
        axis=0
    )


    print(
        "X_stack shape:",
        X_stack.shape
    )


    print(
        "Number of variables:",
        X_stack.shape[0]
    )


    print(
        "Number of spatial pixels:",
        X_stack.shape[1]
        *
        X_stack.shape[2]
    )


    # ========================================================
    # 10.8 Convert to machine learning matrix
    # ========================================================

    n_var = X_stack.shape[0]


    X = X_stack.reshape(
        n_var,
        -1
    ).T


    y = y_data.reshape(
        -1
    )


    mask_flat = lake_mask.reshape(
        -1
    )


    print(
        f"\n[{season}] Building machine learning samples..."
    )


    print(
        "X original shape:",
        X.shape
    )


    print(
        "Y original shape:",
        y.shape
    )


    # ========================================================
    # 10.9 Clean NaN
    # ========================================================

    print(
        f"\n[{season}] Cleaning invalid samples..."
    )


    valid_mask = (

        mask_flat

        &

        np.isfinite(y)

        &

        np.isfinite(X).all(
            axis=1
        )

    )


    valid_count = int(
        valid_mask.sum()
    )


    print(
        "Final valid samples:",
        f"{valid_count:,}"
    )


    # --------------------------------------------------------
    # Print validity of each variable separately
    # --------------------------------------------------------

    print(
        "\nFinal valid sample check:"
    )


    for i, feature in enumerate(
        feature_names
    ):

        count = int(
            np.isfinite(
                X[:, i]
            ).sum()
        )

        print(
            f"{feature:20s}: "
            f"{count:,}"
        )


    if valid_count < 100:

        raise ValueError(

            f"\n{season} final valid samples are only "
            f"{valid_count}, cannot run RF."
        )


    X = X[
        valid_mask
    ]


    y = y[
        valid_mask
    ]


    # ========================================================
    # 10.10 Save raw X/Y
    # ========================================================

    print(
        f"\n[{season}] Saving raw machine learning data..."
    )


    data_file = os.path.join(

        season_out,

        f"{season.lower()}_RF_data.npz"

    )


    np.savez_compressed(

        data_file,

        X=X,

        y=y,

        feature_names=np.asarray(
            feature_names
        )

    )


    print(
        data_file
    )


    # ========================================================
    # 10.11 StandardScaler
    # ========================================================
    #
    # Keep consistent with your successfully running Year code
    #
    # ========================================================

    print(
        f"\n[{season}] StandardScaler..."
    )


    scaler = StandardScaler()


    X_scaled = scaler.fit_transform(
        X
    )


    scaler_file = os.path.join(

        season_out,

        f"{season.lower()}_StandardScaler.joblib"

    )


    joblib.dump(

        scaler,

        scaler_file,

        compress=3

    )


    print(
        f"Scaler saved: "
        f"{scaler_file}"
    )


    # ========================================================
    # 10.12 Train / test split
    # ========================================================

    print(
        f"\n[{season}] Train / Test split..."
    )


    X_train, X_test, y_train, y_test = (

        train_test_split(

            X_scaled,

            y,

            test_size=TEST_SIZE,

            random_state=RANDOM_STATE

        )
    )


    print(
        "Train samples:",
        len(y_train)
    )


    print(
        "Test samples:",
        len(y_test)
    )


    # ========================================================
    # 10.13 Random Forest
    # ========================================================

    print(
        f"\n[{season}] Training Random Forest..."
    )


    model = RandomForestRegressor(

        n_estimators=N_ESTIMATORS,

        max_depth=MAX_DEPTH,

        min_samples_leaf=MIN_SAMPLES_LEAF,

        n_jobs=RF_THREADS,

        random_state=RANDOM_STATE,

        verbose=1
    )


    model.fit(

        X_train,

        y_train

    )


    print(
        f"[{season}] RF training complete"
    )


    # ========================================================
    # 10.14 Test set prediction
    # ========================================================

    print(
        f"\n[{season}] Predicting test set..."
    )


    y_pred = model.predict(
        X_test
    )


    # ========================================================
    # 10.15 Model evaluation
    # ========================================================

    r2 = r2_score(
        y_test,
        y_pred
    )


    rmse = np.sqrt(

        mean_squared_error(

            y_test,

            y_pred

        )
    )


    mae = mean_absolute_error(

        y_test,

        y_pred

    )


    # Pearson
    #
    # Prevent errors in extreme cases

    if (
        np.std(y_test) > 0
        and
        np.std(y_pred) > 0
    ):

        r, p = pearsonr(
            y_test,
            y_pred
        )

    else:

        r = np.nan
        p = np.nan


    # ========================================================
    # 10.16 Output model results
    # ========================================================

    print("\n")
    print(
        "=" * 70
    )

    print(
        f"{season.upper()} MODEL RESULTS"
    )

    print(
        "=" * 70
    )

    print(
        f"R2         = {r2:.6f}"
    )

    print(
        f"RMSE       = {rmse:.6f}"
    )

    print(
        f"MAE        = {mae:.6f}"
    )

    print(
        f"Pearson R  = {r:.6f}"
    )

    print(
        f"P-value    = {p:.6e}"
    )

    print(
        f"Sample Size= {valid_count:,}"
    )

    print(
        "=" * 70
    )


    # ========================================================
    # 10.17 Save performance
    # ========================================================

    performance_df = pd.DataFrame({

        "Season": [season],

        "R2": [r2],

        "RMSE": [rmse],

        "MAE": [mae],

        "Pearson_R": [r],

        "P_value": [p],

        "Sample_Size": [
            valid_count
        ]

    })


    performance_file = os.path.join(

        season_out,

        f"{season.lower()}_model_performance.csv"

    )


    performance_df.to_csv(

        performance_file,

        index=False,

        encoding="utf-8-sig"

    )


    # ========================================================
    # 10.18 Feature Importance
    # ========================================================

    print(
        f"\n[{season}] Feature Importance..."
    )


    importances = (
        model.feature_importances_
    )


    importance_df = pd.DataFrame({

        "feature":
            feature_names,

        "importance":
            importances

    })


    importance_df = (
        importance_df
        .sort_values(
            "importance",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


    # Add percentage
    importance_df[
        "importance_percent"
    ] = (
        importance_df[
            "importance"
        ]
        * 100
    )


    importance_file = os.path.join(

        season_out,

        f"{season.lower()}_feature_importance.csv"

    )


    importance_df.to_csv(

        importance_file,

        index=False,

        encoding="utf-8-sig"

    )


    print(
        "\nVariable importance ranking:"
    )


    print(
        importance_df.to_string(
            index=False
        )
    )


    # ========================================================
    # 10.19 Save model
    # ========================================================

    model_file = os.path.join(

        season_out,

        f"{season.lower()}_RF_model.joblib"

    )


    joblib.dump(

        model,

        model_file,

        compress=3

    )


    print(
        f"\nModel saved:"
    )

    print(
        model_file
    )


    # ========================================================
    # 10.20 Save test set
    # ========================================================

    test_file = os.path.join(

        season_out,

        f"{season.lower()}_RF_test.npz"

    )


    np.savez_compressed(

        test_file,

        X_test=X_test,

        y_test=y_test,

        y_pred=y_pred,

        feature_names=np.asarray(
            feature_names
        )

    )


    print(
        f"Test set saved:"
    )

    print(
        test_file
    )


    # ========================================================
    # 10.21 Save summary for this season
    # ========================================================

    result_file = os.path.join(

        season_out,

        f"{season.lower()}_summary.txt"

    )


    with open(
        result_file,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            f"Season: {season}\n"
        )

        f.write(
            f"Years: {START_YEAR}-{END_YEAR}\n"
        )

        f.write(
            f"Sample Size: {valid_count}\n"
        )

        f.write(
            f"R2: {r2}\n"
        )

        f.write(
            f"RMSE: {rmse}\n"
        )

        f.write(
            f"MAE: {mae}\n"
        )

        f.write(
            f"Pearson_R: {r}\n"
        )

        f.write(
            f"P_value: {p}\n"
        )


    # ========================================================
    # 10.22 Memory cleanup
    # ========================================================

    del X_stack

    del X_arrays

    del X

    del X_scaled

    del y

    del y_train

    del y_test

    del X_train

    del X_test

    del y_pred

    del model

    del scaler

    del y_data

    del lake_mask

    del valid_mask


    gc.collect()


    print(
        f"\n[OK] {season.upper()} complete"
    )


    # ========================================================
    # Return results
    # ========================================================

    return {

        "Season":
            season,

        "R2":
            r2,

        "RMSE":
            rmse,

        "MAE":
            mae,

        "Pearson_R":
            r,

        "P_value":
            p,

        "Sample_Size":
            valid_count

    }


# ============================================================
# 11. Safe run function
# ============================================================

def run_season_safe(season):

    try:

        return run_season(
            season
        )

    except Exception as e:

        print("\n")
        print(
            "#" * 100
        )

        print(
            f"[FAILED] {season.upper()} FAILED"
        )

        print(
            "#" * 100
        )

        print(
            "Error type:",
            type(e).__name__
        )

        print(
            "\nError message:"
        )

        print(
            str(e)
        )

        print(
            "\nFull Traceback:"
        )

        traceback.print_exc()

        print(
            "#" * 100
        )


        return {

            "Season":
                season,

            "R2":
                np.nan,

            "RMSE":
                np.nan,

            "MAE":
                np.nan,

            "Pearson_R":
                np.nan,

            "P_value":
                np.nan,

            "Sample_Size":
                0

        }


# ============================================================
# 12. Main program
# ============================================================

if __name__ == "__main__":


    print("\n")
    print(
        "=" * 100
    )

    print(
        "Starting Seasonal RF Attribution"
    )

    print(
        "=" * 100
    )


    results = []


    # --------------------------------------------------------
    # Run four seasons sequentially
    # --------------------------------------------------------

    for season in SEASONS:


        result = run_season_safe(
            season
        )


        results.append(
            result
        )


        # Recycle memory again after each season

        gc.collect()


    # ========================================================
    # 13. Summarize four-season results
    # ========================================================

    summary_df = pd.DataFrame(
        results
    )


    summary_file = os.path.join(

        base_out,

        "Seasonal_RF_summary.csv"

    )


    summary_df.to_csv(

        summary_file,

        index=False,

        encoding="utf-8-sig"

    )


    # ========================================================
    # 14. Output final results
    # ========================================================

    print("\n")
    print(
        "=" * 100
    )

    print(
        "Four-season RF analysis complete"
    )

    print(
        "=" * 100
    )


    print(
        summary_df.to_string(
            index=False
        )
    )


    print("\n")

    print(
        "Summary file:"
    )

    print(
        summary_file
    )


    print("\n")

    print(
        "All results directory:"
    )

    print(
        base_out
    )


    print("\n")
    print(
        "=" * 100
    )

    print(
        "Program finished"
    )

    print(
        "=" * 100
    )

Global Figures

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from rasterio.warp import reproject, Resampling
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm


# ============================================================
# 0. Font settings
# ============================================================
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 25
plt.rcParams['font.weight'] = 'bold'


# ============================================================
# 1. File paths
# ============================================================

# * clip_1.tif: the only spatial reference
mask_path = r"F:\Reversed_asymmetric_warming_inlake\clip_1.tif"

# * Spring LMLT Trend
tif_path = r"F:\ERA5_Land\LMLT_season\Average\LMLT_average_Trend_Winter.tif"


# ============================================================
# 2. Custom colormap
# ============================================================
colors = [
    "#1f4e79",
    "#4f97c2",
    "#a6cbe3",
    "#f0f0f0",
    "#f2b6a0",
    "#c75d4d",
    "#8c1515"
]

custom_cmap = LinearSegmentedColormap.from_list(
    "nature_diverging",
    colors,
    N=256
)


# ============================================================
# 3. Read clip_1.tif
#
# * clip_1.tif is the only reference
# * Do not resample clip_1.tif
# ============================================================
with rasterio.open(mask_path) as mask_src:

    clip_mask_data = mask_src.read(1)

    reference_shape = mask_src.shape
    reference_transform = mask_src.transform
    reference_crs = mask_src.crs
    reference_bounds = mask_src.bounds

    left = reference_bounds.left
    right = reference_bounds.right
    bottom = reference_bounds.bottom
    top = reference_bounds.top


print("=" * 70)
print("Spatial reference: clip_1.tif")
print("=" * 70)

print("Shape     :", reference_shape)
print("CRS       :", reference_crs)
print("Transform :", reference_transform)
print("Bounds    :", reference_bounds)


# ============================================================
# 4. Read Spring LMLT Trend
# ============================================================
with rasterio.open(tif_path) as src:

    source_data = src.read(1)

    source_transform = src.transform
    source_crs = src.crs
    source_nodata = src.nodata

    print()
    print("=" * 70)
    print("Original Spring LMLT Trend")
    print("=" * 70)

    print("Shape     :", src.shape)
    print("CRS       :", source_crs)
    print("Transform :", source_transform)
    print("NoData    :", source_nodata)


# ============================================================
# 5. Resample Spring LMLT to clip_1.tif
#
# * 721 x 1441 -> 720 x 1440
# * clip_1.tif is not resampled
# * Continuous trend data uses bilinear
# ============================================================

resampled_data = np.full(
    reference_shape,
    np.nan,
    dtype=np.float32
)


if source_nodata is not None:

    reproject(
        source=source_data,
        destination=resampled_data,

        src_transform=source_transform,
        src_crs=source_crs,

        dst_transform=reference_transform,
        dst_crs=reference_crs,

        src_nodata=source_nodata,
        dst_nodata=np.nan,

        resampling=Resampling.bilinear
    )

else:

    reproject(
        source=source_data,
        destination=resampled_data,

        src_transform=source_transform,
        src_crs=source_crs,

        dst_transform=reference_transform,
        dst_crs=reference_crs,

        dst_nodata=np.nan,

        resampling=Resampling.bilinear
    )


# Resampled data
data = resampled_data


print()
print("=" * 70)
print("Resampling complete")
print("=" * 70)

print("Original Spring size :", source_data.shape)
print("Reference size       :", reference_shape)
print("Resampled size       :", data.shape)


# ============================================================
# 6. clip_1.tif spatial mask
#
# * clip_1.tif > 0 -> keep
# * clip_1.tif <= 0 -> mask
# ============================================================

clip_mask = (
    (clip_mask_data <= 0) |
    np.isnan(clip_mask_data)
)


# ============================================================
# 7. Create original LMLT mask
#
# Keep your original mask:
# (1) data == 0
# (2) data < -1000
# (3) NaN
# ============================================================

rows, cols = data.shape

lats = np.linspace(
    top,
    bottom,
    rows
)


mask_data = (
    (data == 0) |
    (data < -1000) |
    np.isnan(data)
)


# ============================================================
# 8. Mask below 60°S
# ============================================================

mask_lat = lats < -60

mask_lat_2d = np.repeat(
    mask_lat[:, np.newaxis],
    cols,
    axis=1
)


# ============================================================
# 9. Merge all masks
#
# * clip_1.tif mask
# * data == 0
# * data < -1000
# * NaN
# * latitude < -60°
# ============================================================

final_mask = (
    clip_mask |
    mask_data |
    mask_lat_2d
)


# ============================================================
# 10. Apply mask
# ============================================================

data = np.ma.masked_array(
    data,
    mask=final_mask
)


# ============================================================
# 11. x10
# ============================================================

data = data * 10


# ============================================================
# 12. Split data by hemisphere and compute mean of positive/negative pixels
# ============================================================

lats_2d = np.repeat(
    lats[:, np.newaxis],
    cols,
    axis=1
)


# Northern Hemisphere
positive_north = data[
    (data > 0) &
    (lats_2d >= 0)
].compressed()

negative_north = data[
    (data < 0) &
    (lats_2d >= 0)
].compressed()


# Southern Hemisphere
positive_south = data[
    (data > 0) &
    (lats_2d < 0)
].compressed()

negative_south = data[
    (data < 0) &
    (lats_2d < 0)
].compressed()


# ============================================================
# 13. Compute means
# ============================================================

mean_positive_north = (
    positive_north.mean()
    if positive_north.size > 0
    else np.nan
)

mean_negative_north = (
    negative_north.mean()
    if negative_north.size > 0
    else np.nan
)

mean_positive_south = (
    positive_south.mean()
    if positive_south.size > 0
    else np.nan
)

mean_negative_south = (
    negative_south.mean()
    if negative_south.size > 0
    else np.nan
)


# ============================================================
# 14. Output statistics
# ============================================================

print()
print("=" * 70)
print("Spring LMLT Trend statistics")
print("=" * 70)

print(
    f"Northern Hemisphere mean positive pixels: "
    f"{mean_positive_north:.3f} °C/decade"
)

print(
    f"Northern Hemisphere mean negative pixels: "
    f"{mean_negative_north:.3f} °C/decade"
)

print(
    f"Southern Hemisphere mean positive pixels: "
    f"{mean_positive_south:.3f} °C/decade"
)

print(
    f"Southern Hemisphere mean negative pixels: "
    f"{mean_negative_south:.3f} °C/decade"
)


# ============================================================
# 15. Create map
# ============================================================

fig = plt.figure(figsize=(10, 5))

ax = plt.axes(
    projection=ccrs.Robinson()
)

ax.set_extent(
    [-180, 180, -60, 90],
    crs=ccrs.PlateCarree()
)

ax.coastlines(
    linewidth=0.8
)


# ============================================================
# 16. Plot data (centered on 0)
# ============================================================

norm = TwoSlopeNorm(
    vmin=-0.2,
    vcenter=0,
    vmax=0.2
)


img = ax.imshow(
    data,
    origin='upper',
    extent=[
        left,
        right,
        bottom,
        top
    ],
    transform=ccrs.PlateCarree(),
    cmap=custom_cmap,
    norm=norm
)


# ============================================================
# 17. Colorbar
# ============================================================

cbar = plt.colorbar(
    img,
    ax=ax,
    orientation='horizontal',
    pad=0.05,
    fraction=0.05
)

cbar.set_label(
    "Trend in LMLT (°C decade⁻¹)"
)


# ============================================================
# 18. Mean value annotations
# ============================================================

# # x_text = 55
# # y_text_pos = -40
# # y_text_neg = -20

# # Northern Hemisphere negative mean
# if not np.isnan(mean_negative_north):
#     ax.text(
#         x_text, y_text_neg,
#         f"N-: {mean_negative_north:.2f}",
#         color="#1f4e79",
#         fontsize=18,
#         fontweight='bold',
#         transform=ccrs.PlateCarree()
#     )

# # Northern Hemisphere positive mean
# if not np.isnan(mean_positive_north):
#     ax.text(a
#         x_text, y_text_pos,
#         f"N+: {mean_positive_north:.2f}",
#         color="#8c1515",
#         fontsize=18,
#         fontweight='bold',
#         transform=ccrs.PlateCarree()
#     )

# # Southern Hemisphere negative mean
# if not np.isnan(mean_negative_south):
#     ax.text(
#         x_text, y_text_neg - 10,
#         f"S-: {mean_negative_south:.2f}",
#         color="#1f4e79",
#         fontsize=18,
#         fontweight='bold',
#         transform=ccrs.PlateCarree()
#     )

# # Southern Hemisphere positive mean
# if not np.isnan(mean_positive_south):
#     ax.text(
#         x_text, y_text_pos - 10,
#         f"S+: {mean_positive_south:.2f}",
#         color="#8c1515",
#         fontsize=18,
#         fontweight='bold',
#         transform=ccrs.PlateCarree()
#     )


# ============================================================
# 19. Show
# ============================================================

plt.tight_layout()
plt.show()